<h1 style="
    font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
    font-size: 36px;
    color: #2c3e50;
    background-color: #ecf0f1;
    padding: 20px;
    border-radius: 12px;
    text-align: center;
    box-shadow: 0px 4px 10px rgba(0, 0, 0, 0.1);">
    Data Analysis and and Visualization of NextGen Outputs

</h1>

**Authors:** 

<ul style="line-height:1.5;">
<li>Ayman Nassar <a href="mailto:ayman.nassar@usu.edu">(ayman.nassar@usu.edu)</a></li>
<li>David Tarboton <a href="mailto:david.tarboton@usu.edu">(david.tarboton@usu.edu)</a></li>
<li>Furqan Baig <a href="mailto:fbaig@illinois.edu">(fbaig@illinois.edu)</a></li>
</ul>

**Last Updated:** 05/15/2026

**Purpose:**

This notebook provides a workflow for **processing, and visualizing NextGen model outputs**. It enables users to examine divide-level (subcatchment) variables, compute watershed-level weighted averages, and generate diagnostic plots to better understand model behavior. It is designed as a companion to the *NextGen Data Preparation* notebook and focuses on analyzing model outputs produced after a successful NextGen simulation.

**Audience:**

Researchers, hydrologists, practitioners, and graduate students working with NextGen hydrologic simulations. Users should be familiar with Python, Jupyter Notebooks, and basic hydrologic modeling concepts.

**Description:**

This notebook requires a directory containing **NextGen simulation outputs**, including variable outputs for each divide (subcatchment), flowpath routing, and other model results. It utilizes geometry and model attributes from the 
<a href="https://communityhydrofabric.s3.us-east-1.amazonaws.com/index.html#hydrofabrics/community/"><strong>HydroFabric Dataset</strong></a> 
to spatially link divide IDs with their corresponding attributes. Weighted watershed averages are computed using the area of each divide to ensure that aggregated time series accurately represent the full watershed behavior.

Utility functions from `hydrofabric_visualization_utils.py` and `ngen_outputs_utilis.py` support the extraction, processing, and visualization of model outputs. This includes generating interactive maps, subcatchment-level variable plots, and aggregated watershed-level hydrographs.

**Data Description:**

This notebook relies on:
- **NextGen model outputs** from a prior simulation and/or calibration 
- **HydroFabric dataset**, including geometry and model attributes  
  (documentation available <a href="https://lynker-spatial.s3-us-west-2.amazonaws.com/hydrofabric/v2.2/hfv2.2-data_model.html"><strong>here</strong></a>)  
- Area-weighted aggregation provides watershed-scale normalized results  

**Software Requirements:**

The notebook uses the following library versions:  
> matplotlib: 3.8.3  
> pandas: 2.2.1  

It also uses code from `hydrofabric_visualization_utils.py` and `ngen_outputs_utilis.py`.


<div style="
    padding: 15px 20px; 
    background-color: #e2f0fe; 
    border-left: 6px solid #3b82f6; 
    color: #1e3a8a; 
    border-radius: 4px; 
    margin-bottom: 20px;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Helvetica, Arial, sans-serif;
">
    <h3 style="margin-top: 0; color: #1e3a8a; font-weight: 700; display: flex; align-items: center; gap: 8px;">
        💡 Quick Note Before You Begin
    </h3>
    <p style="margin-bottom: 10px; font-size: 1.05em;">
        To make sure everything runs smoothly and all settings initialize correctly, <strong>please take a moment to restart the kernel before running the cells below.</strong>
    </p>
    <p style="margin: 0; font-size: 0.95em;">
        <strong>How to do this:</strong> Simply navigate to the <strong>Kernel</strong> menu at the top and select <span style="background-color: rgba(0,0,0,0.05); padding: 2px 6px; border-radius: 3px; border: 1px solid rgba(0,0,0,0.1);"><strong>“Restart Kernel and Clear Outputs of All Cells”</strong></span>. Thank you!
    </p>
</div>

<div style="background:#13294b; border-left:6px solid #5cd6ff; padding:10px 16px; border-radius:8px; margin-top:0; margin-bottom:-6px;">
  <h3 style="margin:0; font-size:20px; font-weight:700; color:#eaf7ff;">
    1. Prepare the Python Environment
  </h3>
</div>

<p style="margin-top:0; margin-bottom:4px; padding-top:0;">
Import all necessary Python libraries and modules. This Jupyter notebook uses two key modules: 
<strong>hydrofabric_visualization_utils</strong>, which provides tools for visualizing hydrofabric datasets, including <code>divides</code>, <code>flowpaths</code>, <code>nexus</code>, and <code>gages</code>, 
and <strong>ngen_outputs_utilis</strong>, which offers functions to aggregate various model outputs from the divide scale to the entire watershed scale, enabling the creation of time-series plots.
</p>

In [ ]:
# ----------------------------- Importing Required Libraries -----------------------------

import pandas as pd
import matplotlib.pyplot as plt
from hydrofabric_visualization_utils import display_hydrofabric_map
from ngen_outputs_utils import ngen_output_analysis, process_usgs_streamflow

<div style="background:#13294b; border-left:6px solid #5cd6ff; padding:10px 16px; border-radius:8px; margin-top:0; margin-bottom:-6px;">
  <h3 style="margin:0; font-size:20px; font-weight:700; color:#eaf7ff;">
    2. Visualization of Hydrofabric Subset
  </h3>
</div>

<p style="margin-top:0; margin-bottom:4px; padding-top:0;">
This interactive map allows users to identify the flowpath ID associated with the downstream watershed. 
If the hydrofabric subset is specified using a USGS gage ID, the map marks the gage location with a pin icon. 
Users can then navigate to the pin and click on it to identify the flowpath ID associated with that gage.
</p>


In [ ]:
# Define your Hydrofabric ID
hydrofabric_id = "gage-10109001"

# Provide the path to your hydrofabric GeoPackage file
gpkg_path = f"/home/jovyan/ngiab_preprocess_output/{hydrofabric_id}/config/{hydrofabric_id}_subset.gpkg"

# Visualize the hydrofabric subset on an interactive map using the GeoPackage file
display_hydrofabric_map(gpkg_path)

<div style="background:#13294b; border-left:6px solid #5cd6ff; padding:10px 16px; border-radius:8px; margin-top:0; margin-bottom:-6px;">
  <h3 style="margin:0; font-size:20px; font-weight:700; color:#eaf7ff;">
    3. Model Outputs Processing and Analysis
  </h3>
</div>

<p style="margin-top:0; margin-bottom:4px; padding-top:0;">
For each variable in the model outputs, the value at each divide (subcatchment) is multiplied by the corresponding subcatchment area and then divided by the total watershed area (area of interest). 
This approach yields a <strong>weighted average</strong> for the entire watershed, normalizing the data so that it accurately reflects the characteristics of the full hydrologic domain.
</p>


<h4 style="background-color:#e6ebff; color:#000000; padding:8px 12px; border-left: 5px solid #6c63ff; border-radius:6px; font-family:'Segoe UI',sans-serif; font-size:16px; margin-top:px;">
🔹 Identifying the Target Flowpath ID for Water Balance Data Analysis
</h4>
<p style="margin-top:0; margin-bottom:6px; padding-top:0;">
Identify the <strong>flowpath ID</strong> associated with the downstream end of the entire watershed. 
This ID represents the terminal/outlet flowpath for the hydrofabric subset and is required for watershed-level model output aggregation.
</p>

<div style="border-left: 6px solid #3b82f6; background:#f0f7ff; padding:12px 14px; border-radius:8px;">
<strong>💡 Tip:</strong> In the interactive map, flowpath IDs appear with a prefix such as <code>wb-2861391</code>.<br>
When specifying the downstream flowpath ID (<code>ds_flowpath_id</code>) in the code cells below, use **only the numeric portion** (e.g., <strong>2861391</strong>) and **omit the <code>wb-</code> prefix**.
</div>


In [ ]:
# Identify the flowpath ID associated with downstream of the entire watershed.
ds_flowpath_id = 2861391  # Downstream flowpath ID
# ds_flowpath_id = 2861474  # Upstream flowpath ID

The table below presents the model outputs, organized into three columns: the variable name (as used by the model), a description of each variable, and the corresponding units.

<h3 style="text-align:center; font-size:18px; margin: 0px 0;"><strong>NextGen Model Outputs</strong></h3>

<table style="border-collapse: collapse; width: 100%; font-size: 14px; text-align:left;">
  <thead>
    <tr style="background-color: navy; color: white; text-align:left;">
      <th style="padding: 8px; text-align:left;">Variable Name</th>
      <th style="padding: 8px; text-align:left;">Description</th>
      <th style="padding: 8px; text-align:left;">Units</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="padding: 6px; text-align:left;">RAIN_RATE</td><td style="padding: 6px; text-align:left;">Rainfall rate (CFE)</td><td style="padding: 6px; text-align:left;">m</td></tr>
    <tr><td style="padding: 6px; text-align:left;">GIUH_RUNOFF</td><td style="padding: 6px; text-align:left;">Lagged and attenuated runoff via GIUH (CFE)</td><td style="padding: 6px; text-align:left;">m</td></tr>
    <tr><td style="padding: 6px; text-align:left;">INFILTRATION_EXCESS</td><td style="padding: 6px; text-align:left;">Rainfall exceeding infiltration capacity (CFE)</td><td style="padding: 6px; text-align:left;">m</td></tr>
    <tr><td style="padding: 6px; text-align:left;">DIRECT_RUNOFF</td><td style="padding: 6px; text-align:left;">Surface runoff from rainfall / throughfall (CFE)</td><td style="padding: 6px; text-align:left;">m</td></tr>
    <tr><td style="padding: 6px; text-align:left;">NASH_LATERAL_RUNOFF</td><td style="padding: 6px; text-align:left;">Lateral subsurface flow (Nash cascade) (CFE)</td><td style="padding: 6px; text-align:left;">m</td></tr>
    <tr><td style="padding: 6px; text-align:left;">DEEP_GW_TO_CHANNEL_FLUX</td><td style="padding: 6px; text-align:left;">Deep groundwater flux to channel (CFE)</td><td style="padding: 6px; text-align:left;">m</td></tr>
    <tr><td style="padding: 6px; text-align:left;">SOIL_TO_GW_FLUX</td><td style="padding: 6px; text-align:left;">Flux from soil to groundwater (CFE)</td><td style="padding: 6px; text-align:left;">m</td></tr>
    <tr><td style="padding: 6px; text-align:left;">Q_OUT</td><td style="padding: 6px; text-align:left;">Catchment outflow (CFE)</td><td style="padding: 6px; text-align:left;">m</td></tr>
    <tr><td style="padding: 6px; text-align:left;">POTENTIAL_ET</td><td style="padding: 6px; text-align:left;">Potential evapotranspiration (CFE)</td><td style="padding: 6px; text-align:left;">m</td></tr>
    <tr><td style="padding: 6px; text-align:left;">ACTUAL_ET</td><td style="padding: 6px; text-align:left;">Actual evapotranspiration (CFE)</td><td style="padding: 6px; text-align:left;">m</td></tr>
    <tr><td style="padding: 6px; text-align:left;">GW_STORAGE</td><td style="padding: 6px; text-align:left;">Groundwater storage (CFE)</td><td style="padding: 6px; text-align:left;">m</td></tr>
    <tr><td style="padding: 6px; text-align:left;">SOIL_STORAGE</td><td style="padding: 6px; text-align:left;">Soil moisture storage (CFE)</td><td style="padding: 6px; text-align:left;">m</td></tr>
    <tr><td style="padding: 6px; text-align:left;">SOIL_STORAGE_CHANGE</td><td style="padding: 6px; text-align:left;">Change in soil moisture storage (CFE)</td><td style="padding: 6px; text-align:left;">m</td></tr>
    <tr><td style="padding: 6px; text-align:left;">SURF_RUNOFF_SCHEME</td><td style="padding: 6px; text-align:left;">Selected surface runoff scheme (CFE)</td><td style="padding: 6px; text-align:left;">Unitless</td></tr>
    <tr><td style="padding: 6px; text-align:left;">NWM_PONDED_DEPTH</td><td style="padding: 6px; text-align:left;">Ponded water depth</td><td style="padding: 6px; text-align:left;">m</td></tr>
    <tr><td style="padding: 6px; text-align:left;">APCP_surface</td><td style="padding: 6px; text-align:left;">Total precipitation rate (AORC)</td><td style="padding: 6px; text-align:left;">mm/hr</td></tr>
    <tr><td style="padding: 6px; text-align:left;">QINSUR</td><td style="padding: 6px; text-align:left;">Total liquid water input to surface rate (Noah-OWP)</td><td style="padding: 6px; text-align:left;">m/s → m</td></tr>
    <tr><td style="padding: 6px; text-align:left;">SNEQV</td><td style="padding: 6px; text-align:left;">Snow water equivalent (Noah-OWP)</td><td style="padding: 6px; text-align:left;">mm (converted to m)</td></tr>
    <tr><td style="padding: 6px; text-align:left;">SNOWH</td><td style="padding: 6px; text-align:left;">Snow depth (Noah-OWP)</td><td style="padding: 6px; text-align:left;">m</td></tr>
    <tr><td style="padding: 6px; text-align:left;">QSNOW</td><td style="padding: 6px; text-align:left;">Snowfall rate on the ground (Noah-OWP)</td><td style="padding: 6px; text-align:left;">mm/s (converted to m)</td></tr>
    <tr><td style="padding: 6px; text-align:left;">ACSNOM</td><td style="padding: 6px; text-align:left;">Accumulated meltwater from bottom snow layer (Noah-OWP)</td><td style="padding: 6px; text-align:left;">mm (converted to m)</td></tr>
    <tr><td style="padding: 6px; text-align:left;">ECAN</td><td style="padding: 6px; text-align:left;">Canopy evaporation (Noah-OWP)</td><td style="padding: 6px; text-align:left;">mm (converted to m)</td></tr>
    <tr><td style="padding: 6px; text-align:left;">ETRAN</td><td style="padding: 6px; text-align:left;">Evaporation of intercepted water (Noah-OWP)</td><td style="padding: 6px; text-align:left;">mm (converted to m)</td></tr>
    <tr><td style="padding: 6px; text-align:left;">QSEVA</td><td style="padding: 6px; text-align:left;">Evaporation rate (Noah-OWP)</td><td style="padding: 6px; text-align:left;">mm/s (converted to m)</td></tr>
    <tr><td style="padding: 6px; text-align:left;">EVAPOTRANS</td><td style="padding: 6px; text-align:left;">Evapotranspiration rate (Noah-OWP)</td><td style="padding: 6px; text-align:left;">m/s (converted to m)</td></tr>
    <tr><td style="padding: 6px; text-align:left;">QRAIN</td><td style="padding: 6px; text-align:left;">Rainfall rate on the ground (Noah-OWP)</td><td style="padding: 6px; text-align:left;">mm/s (converted to m)</td></tr>
    <tr><td style="padding: 6px; text-align:left;">CMC</td><td style="padding: 6px; text-align:left;">Total canopy water (liquid + ice) (Noah-OWP)</td><td style="padding: 6px; text-align:left;">mm (converted to m)</td></tr>
    <tr><td style="padding: 6px; text-align:left;">SNLIQ</td><td style="padding: 6px; text-align:left;">Snow layer liquid water (Noah-OWP)</td><td style="padding: 6px; text-align:left;">mm (converted to m)</td></tr>
    <tr><td style="padding: 6px; text-align:left;">FSNO</td><td style="padding: 6px; text-align:left;">Snow-cover fraction on the ground (Noah-OWP)</td><td style="padding: 6px; text-align:left;">Unitless</td></tr>
  </tbody>
</table>


<h4 style="background-color:#e6ebff; color:#000000; padding:6px 10px; border-left: 5px solid #6c63ff; border-radius:6px; font-family:'Segoe UI',sans-serif; font-size:15px;">
🔹 Processing and Aggregating Model Outputs
</h4>
For each variable, the value at each divide (subcatchment) is multiplied by the corresponding subcatchment area and then divided by the total area of the watershed (area of interest), providing a weighted average for the entire watershed. Additionally, the model's flow data, stored in the 'troute' NetCDF file, is extracted based on the downstream flowpath ID. The function used to process and aggregate the model ouputs is `ngen_output_analysis`, which generates a CSV file that compiles all the processed data

In [ ]:
# Call the function to process the data and generate the CSV
ngen_agg_outputs = ngen_output_analysis(hydrofabric_id, ds_flowpath_id)
ngen_agg_outputs

In [ ]:
# Display all variable (column) names available in the aggregated outputs table
ngen_agg_outputs.columns

In [ ]:
ngen_agg_outputs = ngen_agg_outputs[ngen_agg_outputs['Time'] > '2020-10-01 01:00:00']

<h4 style="background-color:#e6ebff; color:#000000; padding:6px 10px; border-left: 5px solid #6c63ff; border-radius:6px; font-family:'Segoe UI',sans-serif; font-size:15px;">
🔹 Additional Derived Variables
</h4>
The following derived variables are generated using combinations of the main hydrologic outputs above. This is  useful for additional interpretation, integrated comparisons between fluxes and storages, and water balance component evaluation across time.

In [ ]:
# ---- Derived diagnostic variables ----

# Combined total snow + canopy water mass (solid + liquid) [m]
ngen_agg_outputs["SNEQV_SNLIC_CMC"] = (
    ngen_agg_outputs["SNEQV_m"] +
    ngen_agg_outputs["SNLIQ_m"] +
    ngen_agg_outputs["CMC_m"]
)

# Change in total snow+liquid+canopy water between timesteps [m]
ngen_agg_outputs["DIFF_SNEQV_SNLIC_CMC"] = ngen_agg_outputs["SNEQV_SNLIC_CMC"].diff()

# Estimated the cumulative inferred NOAH-OWP upward flux (sublimation)
ngen_agg_outputs["E_OWP"] = (
    ngen_agg_outputs["APCP_surface_m"] -
    ngen_agg_outputs["RAIN_RATE_m"] -
    ngen_agg_outputs["DIFF_SNEQV_SNLIC_CMC"]
)

ngen_agg_outputs


<h4 style="background-color:#e6ebff; color:#000000; padding:6px 10px; border-left: 5px solid #6c63ff; border-radius:6px; font-family:'Segoe UI',sans-serif; font-size:15px;">
🔹 Retrieve the Observed Streamflow Data from the USGS
</h4>
In this step, we retrieve the observed streamflow time-series data directly from the USGS National Water Information System (NWIS). These measurements represent the real-world discharge at the selected gage location and will be used to compare against the simulated streamflow produced by the NextGen model for validation and performance assessment.

In [ ]:
# Retrieve the observed streamflow data from the USGS

site = "10109001"     # Specify the USGS gage ID
start = "2017-10-01"  # Specify the start time
end = "2021-09-30"    # Specify the end time

# Provide the path to your hydrofabric GeoPackage file
# The hydrofabric subset is required to retrieve the watershed area, which is essential for calculating streamflow in (m/hr),
# as all model output variables are expressed in m/hr.
gpkg_path = f"/home/jovyan/ngiab_preprocess_output/{hydrofabric_id}/config/{hydrofabric_id}_subset.gpkg"
print(gpkg_path)
# The process_usgs_streamflow function is used to retrieve the observed USGS streamflow measurements and process the data, 
# converting it to units of (m/hr).
df_str_usgs = process_usgs_streamflow(site, start, end, gpkg_path)

df_str_usgs

<h4 style="background-color:#e6ebff; color:#000000; padding:6px 10px; border-left: 5px solid #6c63ff; border-radius:6px; font-family:'Segoe UI',sans-serif; font-size:15px; margin-top:0px; margin-bottom:4px;">
🔹 Create a Combined DataFrame for Observed vs Simulated Streamflow
</h4>
This step merges both the simulated streamflow from the NextGen model and the observed USGS streamflow into one unified DataFrame to enable direct comparison and plotting.


In [ ]:
df_stream_compare = pd.DataFrame({"Time": ngen_agg_outputs.Time, "Simulated_m": ngen_agg_outputs.Q_OUT_m, "Observed_m": df_str_usgs["Streamflow (m/hr)"]})
df_stream_compare

<h4 style="background-color:#e6ebff; color:#000000; padding:6px 10px; border-left: 5px solid #6c63ff; border-radius:6px; font-family:'Segoe UI',sans-serif; font-size:15px; margin-top:0px; margin-bottom:4px;">
🔹 Plot Observed vs. Simulated Streamflow Time Series
</h4>
This plot compares the simulated streamflow produced by the NextGen model against the observed USGS streamflow measurements over the same period.

In [ ]:
fig = plt.figure(figsize=(12, 6))
ax = plt.gca()

# Ensure datetime + sorted
df_stream_compare = df_stream_compare.copy()
df_stream_compare["Time"] = pd.to_datetime(df_stream_compare["Time"]).dt.tz_localize(None)
df_stream_compare = df_stream_compare.sort_values("Time")

# Plot observed vs simulated from the combined dataframe
ax.plot(df_stream_compare["Time"], df_stream_compare["Observed_m"],  linewidth=2, linestyle='-',  label="Observed Streamflow (USGS)")
ax.plot(df_stream_compare["Time"], df_stream_compare["Simulated_m"], linewidth=2, linestyle='--', label="Simulated Streamflow (NextGen)")

# Axes + title
ax.set_title("Observed vs. Simulated Streamflow", fontsize=16, fontweight='bold')
ax.set_xlabel("Time", fontsize=14, fontweight='bold')
ax.set_ylabel("Streamflow (m/hr)", fontsize=14, fontweight='bold')

# ----- X limits and Month–Year ticks (rotated 45°), same style as your other cell -----
import matplotlib.dates as mdates
tmin, tmax = df_stream_compare["Time"].min(), df_stream_compare["Time"].max()
ax.set_xlim(tmin, tmax)

# Smart monthly spacing (~24 ticks target)
months_span   = (tmax.year - tmin.year) * 12 + (tmax.month - tmin.month) + 1
target_ticks  = 24
interval      = max(1, months_span // target_ticks)

ax.xaxis.set_major_locator(mdates.MonthLocator(interval=interval))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.xticks(rotation=45)

# Grid + legend
ax.grid(True, linestyle='--', alpha=0.7)
ax.legend(loc='upper right', fontsize=12)

# Save, then show
output_path = "/home/jovyan/streamflow_pre_cal_plot.png"
plt.tight_layout()
plt.savefig(output_path, dpi=200)
print(output_path)

plt.show()


<h4 style="background-color:#e6ebff; color:#000000; padding:6px 10px; border-left: 5px solid #6c63ff; border-radius:6px; font-family:'Segoe UI',sans-serif; font-size:15px; margin-top:0px; margin-bottom:4px;">
🔹 Calculate Performance Metrics (KGE & RMSE)
</h4>
In this step, two standard hydrologic model evaluation metrics are computed to assess how well the simulated streamflow matches the observed USGS streamflow. The Root Mean Square Error (RMSE) reflects overall magnitude error, while the Kling-Gupta Efficiency (KGE) evaluates correlation, variability, and bias performance between the simulated and observed time series.


In [ ]:
from sklearn.metrics import mean_squared_error
import numpy as np

# clean data (drop rows where either is NaN)
df_clean = df_stream_compare.dropna(subset=["Observed_m","Simulated_m"])

obs = df_clean["Observed_m"].values
sim = df_clean["Simulated_m"].values

# RMSE
rmse = np.sqrt(mean_squared_error(obs, sim))

# KGE
r = np.corrcoef(obs, sim)[0,1]
alpha = np.std(sim) / np.std(obs)
beta = np.mean(sim) / np.mean(obs)
kge = 1 - np.sqrt((r-1)**2 + (alpha-1)**2 + (beta-1)**2)

print("RMSE =", rmse)
print("KGE =", kge)

<h4 style="background-color:#e6ebff; color:#000000; padding:6px 10px; border-left: 5px solid #6c63ff; border-radius:6px; font-family:'Segoe UI',sans-serif; font-size:15px; margin-top:0px; margin-bottom:4px;">
🔹 Select Key Hydrologic Variables for Plotting
</h4>
In this step, we define the primary hydrologic variables to be included in the cumulative time series plots. These selected variables represent the major components of the water balance system, including precipitation inputs, modeled streamflow output, evapotranspiration, groundwater and soil storage, as well as snow-related storage components derived from Noah-OWP.

In [ ]:
# Define the set of key model output variables that will be included in the plots
variables_to_plot = [
    "APCP_surface_m",   # AORC precipitation (m)
    "RAIN_RATE_m",      # NextGen rain rate (m)
    "Q_OUT_m",          # Simulated streamflow outflow (m)
    "ACTUAL_ET_m",      # Actual evapotranspiration (m)
    "GW_STORAGE_m",     # Groundwater storage (m)
    "SOIL_STORAGE_m",   # Soil moisture storage (m)
    "SNEQV_SNLIC_CMC",  # NOAH-OWP Storage (Snow Water Equivalent + Liquid + Canopy) (m)
    "E_OWP"             # ET from Noah-OWP (converted to m)
]


<h4 style="background-color:#e6ebff; color:#000000; padding:6px 10px; border-left: 5px solid #6c63ff; border-radius:6px; font-family:'Segoe UI',sans-serif; font-size:15px; margin-top:0px; margin-bottom:4px;">
🔹 Cumulative Time-Series Plot of Hydrologic Variables
</h4>
This step generates cumulative time-series plots for the selected model output variables. Visualizing fluxes and storage components in cumulative form helps reveal watershed-scale behavior, process contributions, and the temporal role of precipitation, evapotranspiration, runoff, and storage changes over the simulation period.


In [ ]:
# --- Setup ---
# %matplotlib inline  # uncomment in Jupyter if needed
import itertools
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd

# ------------------------------------------------------------------
# Source data
# ------------------------------------------------------------------
# If you already have ngen_agg_outputs and df_str_usgs in memory, keep these lines.
data = ngen_agg_outputs.copy()

# Ensure the "Time" column is datetime and sorted
data["Time"] = pd.to_datetime(data["Time"])
data = data.sort_values("Time")
# data = data[data['Time'] > '2019-10-01 01:00:00']
# Load/align USGS streamflow (df_str_usgs should already exist)
df_str_usgs["Time"] = pd.to_datetime(df_str_usgs["Time"]).dt.tz_localize(None)
data["Time"]       = pd.to_datetime(data["Time"]).dt.tz_localize(None)

# Merge USGS streamflow onto model dataframe on Time
merged_df = pd.merge(
    data,
    df_str_usgs[["Time", "Streamflow (m/hr)"]],
    on="Time",
    how="left"
)

# Build the cumulative observed series
strflow_usgs = merged_df[["Time", "Streamflow (m/hr)"]].copy()
strflow_usgs["Observed (USGS) Cumulative Streamflow (m/hr)"] = strflow_usgs["Streamflow (m/hr)"].cumsum()

# ------------------------------------------------------------------
# Variables you want to plot
# ------------------------------------------------------------------
# Example:
# variables_to_plot = ["APCP_surface_m", "RAIN_RATE_m", "ACTUAL_ET_m", "Q_OUT_m",
#                      "GW_STORAGE_m", "SOIL_STORAGE_m", "SNEQV_SNLIC_CMC", "E_OWP"]
# Make sure variables_to_plot exists before running this cell.

available_vars = [v for v in variables_to_plot if v in data.columns]
missing_vars = [v for v in variables_to_plot if v not in data.columns]
if missing_vars:
    print(f"Skipping missing variables: {missing_vars}")
if not available_vars:
    raise ValueError("None of the requested variables are present in the dataframe.")

# Vars that should NOT be cumulatively summed (storages)
non_cumulative_vars = {"GW_STORAGE_m", "SOIL_STORAGE_m", "SNEQV_SNLIC_CMC"}

# ------------------------------------------------------------------
# Legend label mapping (your requested names)
# ------------------------------------------------------------------
label_map = {
    "APCP_surface_m": "Cumulative AORC Precipitation",
    "RAIN_RATE_m": "Cumulative Surface Water Input passed from NOAH-OWP to CFE",
    "Q_OUT_m": "Cumulative Modeled Outflow",
    "ACTUAL_ET_m": "Cumulative Modeled Actual ET",
    "GW_STORAGE_m": "CFE Groundwater Storage",
    "SOIL_STORAGE_m": "CFE Soil Storage",
    "SNEQV_SNLIC_CMC": "NOAH-OWP Storage (Snow Water Equivalent + Liquid + Canopy)",
    "E_OWP": "Cumulative inferred NOAH-OWP upward flux (sublimation)",
    # Observed line handled separately below
}

# Colors (cycle safely even if you have many variables)
color_cycle = itertools.cycle(
    ['tab:blue', 'tab:orange', 'tab:green', 'tab:red',
     'tab:purple', 'tab:brown', 'tab:pink', 'tab:olive', 'tab:cyan']
)

# --- Plot ---
fig = plt.figure(figsize=(14, 10))
ax = plt.gca()

plotted_max = []   # track maxima for y-limits

for i, var in enumerate(available_vars):
    series = pd.to_numeric(data[var], errors="coerce")

    # cumulative vs non-cumulative
    if var in non_cumulative_vars:
        y = series
    else:
        y = series.cumsum()

    # Only plot if there is at least some non-NaN data
    if y.notna().any():
        c = next(color_cycle)
        lbl = label_map.get(var, f"Cumulative {var}" if var not in non_cumulative_vars else var)
        line = ax.plot(
            data["Time"], y,
            label=lbl,
            color=c,
            linestyle='-' if i % 2 == 0 else '--',
            linewidth=2
        )[0]
        # Store finite values to compute y-limit later
        finite_vals = y[np.isfinite(y)]
        if not finite_vals.empty:
            plotted_max.append(finite_vals.max())

# Plot the cumulative observed USGS streamflow with your custom label
if "Observed (USGS) Cumulative Streamflow (m/hr)" in strflow_usgs:
    y_obs = pd.to_numeric(strflow_usgs["Observed (USGS) Cumulative Streamflow (m/hr)"], errors="coerce")
    if y_obs.notna().any():
        line_obs = ax.plot(
            strflow_usgs["Time"],
            y_obs,
            label="Cumulative Observed Streamflow",
            linewidth=2
        )[0]
        finite_obs = y_obs[np.isfinite(y_obs)]
        if not finite_obs.empty:
            plotted_max.append(finite_obs.max())

# Titles / labels
ax.set_title('Cumulative Time Series of Hydrologic Variables', fontsize=16, fontweight='bold', color='darkblue')
ax.set_xlabel('Date/Time', fontsize=14, fontweight='bold')
ax.set_ylabel('Value (m)', fontsize=14, fontweight='bold')

# Grid
ax.grid(True, which='both', axis='both', linestyle='--', color='lightgray', alpha=0.7)

# ----- X limits and Month-Year ticks (rotated 45°) -----
time_min, time_max = data["Time"].min(), data["Time"].max()
ax.set_xlim(time_min, time_max)

# Smart monthly spacing similar to your example snippet
start = time_min
end   = time_max
months_span = (end.year - start.year) * 12 + (end.month - start.month) + 1
target_ticks = 24
interval = max(1, months_span // target_ticks)

ax.xaxis.set_major_locator(mdates.MonthLocator(interval=interval))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.xticks(rotation=45)

# Y limits from plotted data
if plotted_max:
    ymax = max([m for m in plotted_max if pd.notna(m) and np.isfinite(m)])
    if pd.notna(ymax) and np.isfinite(ymax):
        ax.set_ylim(0, ymax * 1.1)

# Single, clean legend
handles, labels = ax.get_legend_handles_labels()
if handles:
    ax.legend(handles, labels, loc='upper left', fontsize=12)

plt.tight_layout()

# Save
output_path = "/home/jovyan/WB_pre_cal_plot.png"
plt.savefig(output_path, dpi=150)
print(output_path)

plt.show()
